# 07 · Mixture-of-experts routing: top-k, dispatch, load balancing

**Needed for:** DeepSeekMoE (LLM track, step 5) and DeepSeek-V3 (step 7). Read the Switch Transformer paper
first for the original load-balancing loss.

An MoE layer replaces one big feed-forward network with $E$ smaller ones ("experts") and a **router** that sends
each token to $k$ of them. Parameters grow with $E$, compute per token grows only with $k$.

The hard parts are all bookkeeping:
1. **Dispatch:** group tokens by expert so each expert runs one batched matmul.
2. **Combine:** scatter expert outputs back to token order, weighted by the gate values.
3. **Balance:** without pressure, the router collapses onto a few experts and the rest never train.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Top-k routing

Two common gate functions:
- **Softmax then top-k** (Switch, Mixtral, DeepSeekMoE): probabilities over all experts, keep the $k$ largest,
  optionally renormalize them to sum to 1.
- **Sigmoid then top-k, renormalized** (DeepSeek-V3): each expert gets an independent affinity score in (0, 1).

`bias` is used only to *choose* experts, never to weight them. That's the aux-loss-free trick in section 4.

In [ ]:
def route(logits, k, gate="softmax", bias=None):
    """logits: (N, E). Returns (weights, idx), both (N, k)."""
    scores = logits.softmax(-1) if gate == "softmax" else logits.sigmoid()
    choice = scores if bias is None else scores + bias
    idx = choice.topk(k, dim=-1).indices
    weights = scores.gather(-1, idx)
    weights = weights / weights.sum(-1, keepdim=True)
    return weights, idx


def test_route(fn):
    logits = torch.tensor([[0.0, 3.0, 1.0, 2.0], [5.0, 0.0, 0.0, 4.0]])
    w, idx = fn(logits, 2)
    assert idx.tolist() == [[1, 3], [0, 3]]
    torch.testing.assert_close(w.sum(-1), torch.ones(2))
    assert (w[:, 0] >= w[:, 1]).all()
    _, idx_b = fn(logits, 2, bias=torch.tensor([0.0, 0.0, 10.0, 0.0]))
    assert (idx_b == 2).any(-1).all(), "bias changes which experts are chosen"


test_route(route)
print("routing ok")

## 2. Dispatch and combine

The readable way: loop over experts, find the (token, slot) pairs routed to each, run the expert on just those
tokens, and `index_add_` the weighted outputs back. Production kernels (grouped GEMM, DeepEP) do the same thing
with a sort instead of a Python loop.

Check it against a dense reference that runs *every* expert on *every* token and masks.

In [ ]:
class Expert(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.w1, self.w3, self.w2 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


def moe_forward(x, experts, weights, idx):
    """x: (N, d); weights, idx: (N, k). Returns (N, d)."""
    out = torch.zeros_like(x)
    for e, expert in enumerate(experts):
        token_ids, slot = (idx == e).nonzero(as_tuple=True)       # which tokens chose e, and in which slot
        if token_ids.numel() == 0:
            continue
        y = expert(x[token_ids]) * weights[token_ids, slot, None]
        out.index_add_(0, token_ids, y)
    return out


def moe_dense_reference(x, experts, weights, idx):
    all_out = torch.stack([ex(x) for ex in experts], dim=1)          # (N, E, d)
    gate = torch.zeros(x.shape[0], len(experts)).scatter(1, idx, weights)
    return (gate[..., None] * all_out).sum(1)


def test_moe_forward(fn):
    torch.manual_seed(0)
    N, d, E, k = 50, 16, 8, 2
    experts = nn.ModuleList(Expert(d, 32) for _ in range(E))
    x = torch.randn(N, d)
    w, idx = route(torch.randn(N, E), k)
    torch.testing.assert_close(fn(x, experts, w, idx), moe_dense_reference(x, experts, w, idx))


test_moe_forward(moe_forward)
print("sparse dispatch matches the dense reference")

## 3. Load: capacity factor and the auxiliary loss

With expert parallelism each expert lives on a different GPU with a fixed buffer. **Capacity** =
`capacity_factor * N * k / E` tokens; tokens beyond it are *dropped* (they skip the layer via the residual).

**Switch Transformer's auxiliary loss** pushes the router toward uniform load:
$$\mathcal{L}_{aux} = E \sum_{i=1}^{E} f_i \, P_i$$
$f_i$ = fraction of token assignments that went to expert $i$ (not differentiable), $P_i$ = mean router
probability for expert $i$ (differentiable). It equals 1 when perfectly balanced and grows with imbalance.

In [ ]:
def load_fractions(idx, E):
    return torch.bincount(idx.flatten(), minlength=E).float() / idx.numel()


def aux_loss(logits, idx):
    E = logits.shape[-1]
    return E * (load_fractions(idx, E) * logits.softmax(-1).mean(0)).sum()


def dropped_tokens(idx, E, capacity_factor=1.25):
    cap = int(capacity_factor * idx.numel() / E)
    counts = torch.bincount(idx.flatten(), minlength=E)
    return (counts - cap).clamp(min=0).sum().item()


N, E, k = 4096, 16, 2
balanced = torch.randn(N, E) * 0.01
skewed = torch.randn(N, E) + torch.linspace(0, 3, E)     # later experts preferred
for name, lg in (("balanced", balanced), ("skewed", skewed)):
    _, idx = route(lg, k)
    print(f"{name:9s} aux loss {aux_loss(lg, idx).item():.3f}   max load / mean load "
          f"{(load_fractions(idx, E).max() * E).item():.2f}   dropped at CF 1.25: {dropped_tokens(idx, E)}")

## 4. Aux-loss-free balancing (DeepSeek-V3)

An auxiliary loss fights the main loss: its gradient distorts the router. DeepSeek-V3 instead keeps a per-expert
**bias** that is added to the scores only when choosing the top-k, and after each step nudges it:

$$b_i \leftarrow b_i + \gamma \cdot \text{sign}(\bar{c} - c_i)$$

where $c_i$ is expert $i$'s load and $\bar c$ the mean. Overloaded experts get pushed down, idle ones pulled up.
No gradient involved. (V3 still adds a very small sequence-level aux loss as a safety net.)

Below, a router that strongly prefers some experts is left frozen; only the bias adapts.

In [ ]:
def update_bias(bias, idx, E, gamma=1e-3):
    load = torch.bincount(idx.flatten(), minlength=E).float()
    return bias + gamma * torch.sign(load.mean() - load)


def test_update_bias(fn):
    idx = torch.tensor([[0, 1], [0, 1], [0, 2]])        # loads: e0=3, e1=2, e2=1, e3=0 (mean 1.5)
    b = fn(torch.zeros(4), idx, 4, gamma=0.1)
    torch.testing.assert_close(b, torch.tensor([-0.1, -0.1, 0.1, 0.1]))


test_update_bias(update_bias)

torch.manual_seed(0)
d, E, k = 32, 16, 2
W_router = torch.randn(d, E) * 0.3 + torch.linspace(-0.3, 0.3, E)  # built-in preference
bias = torch.zeros(E)
for step in range(3001):
    x = torch.randn(1024, d) + 0.5
    logits = x @ W_router
    _, idx = route(logits, k, gate="sigmoid", bias=bias)
    if step % 1000 == 0:
        print(f"step {step:4d}: max load / mean load = {(load_fractions(idx, E).max() * E).item():.2f}")
    bias = update_bias(bias, idx, E, gamma=1e-3)

## 5. Fine-grained and shared experts (DeepSeekMoE)

DeepSeekMoE makes two changes to the standard "few big experts, top-2" design:
1. **Fine-grained segmentation:** split each expert into $m$ smaller ones and route to $m\times$ more of them.
   Same parameters and FLOPs, but far more possible combinations, so experts specialize more.
2. **Shared experts:** a few experts process *every* token, capturing common knowledge so routed experts
   don't all relearn it.

DeepSeek-V3: 1 shared + 256 routed experts, 8 active per token.

In [ ]:
class DeepSeekMoE(nn.Module):
    def __init__(self, d, n_routed, n_shared, k, expert_hidden):
        super().__init__()
        self.k = k
        self.router = nn.Linear(d, n_routed, bias=False)
        self.routed = nn.ModuleList(Expert(d, expert_hidden) for _ in range(n_routed))
        self.shared = nn.ModuleList(Expert(d, expert_hidden) for _ in range(n_shared))
        self.register_buffer("bias", torch.zeros(n_routed))

    def forward(self, x):
        shape = x.shape
        x = x.reshape(-1, shape[-1])
        weights, idx = route(self.router(x), self.k, gate="sigmoid", bias=self.bias)
        out = moe_forward(x, self.routed, weights, idx)
        for ex in self.shared:
            out = out + ex(x)
        if self.training:
            self.bias = update_bias(self.bias, idx, len(self.routed))
        return out.reshape(shape)


coarse = DeepSeekMoE(d=64, n_routed=8, n_shared=0, k=2, expert_hidden=256)
fine = DeepSeekMoE(d=64, n_routed=32, n_shared=0, k=8, expert_hidden=64)
count = lambda m: sum(p.numel() for p in m.parameters())
print("params  coarse:", count(coarse), "  fine:", count(fine), "(nearly equal: only the router grows)")
print("active hidden units per token  coarse:", 2 * 256, "  fine:", 8 * 64)
print("possible expert combinations  coarse: C(8,2) =", math.comb(8, 2), "  fine: C(32,8) =", math.comb(32, 8))
print(DeepSeekMoE(64, 16, 1, 4, 64)(torch.randn(2, 10, 64)).shape)

## Things to be able to explain

- Why is MoE training communication-heavy? (All-to-all: every GPU sends tokens to every expert's GPU and back.)
- With k = 1, why must you *not* renormalize the gate? (The weight would always be 1, so the router gets no
  gradient. Switch Transformer multiplies by the raw softmax probability instead.)
- What goes wrong at inference when expert load is unbalanced? (One GPU's experts become the bottleneck.)

## Drill

In [ ]:
def my_route(logits, k, gate="softmax", bias=None):
    raise NotImplementedError


def my_moe_forward(x, experts, weights, idx):
    raise NotImplementedError


def my_update_bias(bias, idx, E, gamma=1e-3):
    raise NotImplementedError


drill(test_route, my_route)
drill(test_moe_forward, my_moe_forward)
drill(test_update_bias, my_update_bias)